In [ ]:
import anthropic
import openai
from typing import Literal
from pydantic import BaseModel, create_model
from tqdm import tqdm

In [ ]:
x = ['f', 'l', 'p']

class Response(BaseModel):
    response:  Literal[tuple(x)]

In [ ]:
# Use this dataset instead though please (just a sampling)

In [ ]:
curr_path = "/workspace/psychometrics_for_LLMs/llm_psychometrics/data"
filename = "human_annotations_uuid.json"

In [ ]:
import json
with open(f"{curr_path}/human_annotations_uuid.json", "r") as f_in:
    targets = json.load(f_in)

In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
os.environ['OPENAI_API_KEY'] = 'YOUR_API_KEY_HERE'
os.environ["ANTHROPIC_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
client = openai.Client()
response = client.responses.parse(
    model="gpt-4o-2024-08-06",
    input=[
        {"role": "system", "content": "Help users"},
        {
            "role": "user",
            "content": "Letter after k",
        },
    ],
    text_format=Response,
)

In [ ]:
response.output_parsed

In [ ]:
from datasets import load_dataset

# Load the dataset from HuggingFace
dataset = load_dataset(
    "thoughtworks/psychometric_personas",
    data_files="data/train-00000-of-00001.parquet",
    split="train[0:100]"
)

# Optionally, display the first few rows to verify loading
#print(dataset[17])
dataset

In [ ]:
len(dataset)

In [ ]:
from pydantic import BaseModel, conint, create_model
from anthropic import Anthropic
from tqdm import tqdm  # assuming you're using this
import os
import pickle
import json

# ---------- Pydantic models ----------

# Base schema (still fine to keep around if you use it elsewhere)
class PersonaDatasetQualityReviewBase(BaseModel):
    clarity: conint(ge=0, le=5)
    originality: conint(ge=0, le=5)
    coherence: conint(ge=0, le=5)
    diversity: conint(ge=0, le=5)
    realism: conint(ge=0, le=5)
    psychological_depth: conint(ge=0, le=5)
    consistency: conint(ge=0, le=5)
    informativeness: conint(ge=0, le=5)
    ethical_considerations: conint(ge=0, le=5)
    demographic_fidelity: conint(ge=0, le=5)
    overall_score: conint(ge=0, le=5)

# ---------- Anthropic client ----------

# In real code, prefer: export ANTHROPIC_API_KEY=... and omit api_key param.
client = Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"]
)

# ---------- Main loop ----------

reviews_dict = {}  # or load from pickle/json if you already persist this

SYSTEM_PROMPT = (
    "You are an expert reviewer evaluating the quality of entries in a dataset of police personas. "
    "Your task is to assess the quality of the dataset entry itself, not the competency or character of the persona described. "
    "Rate the following aspects from 0 (worst) to 10 (best) unless otherwise specified:\n"
    "- clarity: Is the persona description clear and understandable?\n"
    "- originality: Does the entry avoid clichés and present a unique character?\n"
    "- coherence: Is the information internally consistent and logically structured?\n"
    "- diversity: Does the persona contribute to a diverse set of police profiles?\n"
    "- realism: Does the persona feel plausible and authentic for a police context?\n"
    "- psychological_depth: Does the entry provide meaningful insight into the persona's inner life, motivations, and psychological complexity? (Focus especially on this metric.)\n"
    "- consistency: Are details about the persona consistent throughout?\n"
    "- informativeness: Does the entry provide rich, relevant information about the persona?\n"
    "- ethical_considerations: Is the entry free from harmful stereotypes or bias?\n"
    "- demographic_fidelity: Is the persona plausible for the demographic data provided "
    "(e.g., a 22 year old should not be described as retiring with decades of experience)?\n"
    "- overall_score: Your overall assessment of the dataset entry's quality.\n"
    "Return only the scores in the specified schema, and include a UID string field (UID) for this entry. "
    "Remember: you are judging the quality of the dataset entry, not the police persona's job performance."
)
for row in tqdm(dataset):  # assuming `dataset` is defined
    uid = row["uuid"]
    if uid in reviews_dict:
        continue  # Skip if already reviewed

    # Dynamically create a schema with the UID set to the current row's UID as a default
    DynamicPersonaDatasetQualityReview = create_model(
        "DynamicPersonaDatasetQualityReview",
        UID=(str, uid),
        clarity=(conint(ge=0, le=5), ...),
        originality=(conint(ge=0, le=5), ...),
        coherence=(conint(ge=0, le=5), ...),
        diversity=(conint(ge=0, le=5), ...),
        realism=(conint(ge=0, le=5), ...),
        psychological_depth=(conint(ge=0, le=5), ...),
        consistency=(conint(ge=0, le=5), ...),
        informativeness=(conint(ge=0, le=5), ...),
        ethical_considerations=(conint(ge=0, le=5), ...),
        demographic_fidelity=(conint(ge=0, le=5), ...),
        overall_score=(conint(ge=0, le=5), ...),
        __base__=BaseModel,
    )

    # Pydantic v2 compatibility
    if hasattr(DynamicPersonaDatasetQualityReview, "model_rebuild"):
        DynamicPersonaDatasetQualityReview.model_rebuild()

    # ----- Anthropic structured output call -----
    response = client.beta.messages.parse(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        betas=["structured-outputs-2025-11-13"],
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": row["persona_string"],
            }
        ],
        output_format=DynamicPersonaDatasetQualityReview,
    )

    persona_review = response.parsed_output
    # Ensure the UID is set correctly (in case the model changes it)
    persona_review.UID = uid

    # Upsert into the dictionary using UID as the key
    reviews_dict[uid] = persona_review

# Example: save to JSON (if you want a serializable version)
# (you may need .model_dump() if using Pydantic v2)
serializable_reviews = {
    uid: review.model_dump() for uid, review in reviews_dict.items()
}

with open("anthropic_persona_reviews.json", "w", encoding="utf-8") as f:
    json.dump(serializable_reviews, f, ensure_ascii=False, indent=2)


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, conint, create_model

# The base schema for dynamic creation
class PersonaDatasetQualityReviewBase(BaseModel):
    clarity: conint(ge=0, le=5)
    originality: conint(ge=0, le=5)
    coherence: conint(ge=0, le=5)
    diversity: conint(ge=0, le=5)
    realism: conint(ge=0, le=5)
    psychological_depth: conint(ge=0, le=5)
    consistency: conint(ge=0, le=5)
    informativeness: conint(ge=0, le=5)
    ethical_considerations: conint(ge=0, le=5)
    demographic_fidelity: conint(ge=0, le=5)
    overall_score: conint(ge=0, le=5)
    # notes: Optional[str]  # Optional notes or comments about the review

client = OpenAI()

import os
import pickle

# Try to load the reviews dictionary from disk, or create a new one if not present
reviews_dict = {}

for row in tqdm(dataset):  # Remove the select(range(2)) to review all entries
    uid = row["uuid"]
    if uid in reviews_dict:
        continue  # Skip if already reviewed

    # Dynamically create a schema with the UID set to the current row's UID as a default
    DynamicPersonaDatasetQualityReview = create_model(
        'DynamicPersonaDatasetQualityReview',
        UID=(str, uid),
        clarity=(conint(ge=0, le=10), ...),
        originality=(conint(ge=0, le=10), ...),
        coherence=(conint(ge=0, le=10), ...),
        diversity=(conint(ge=0, le=10), ...),
        realism=(conint(ge=0, le=10), ...),
        psychological_depth=(conint(ge=0, le=10), ...),
        consistency=(conint(ge=0, le=10), ...),
        informativeness=(conint(ge=0, le=10), ...),
        ethical_considerations=(conint(ge=0, le=10), ...),
        demographic_fidelity=(conint(ge=0, le=10), ...),
        overall_score=(conint(ge=0, le=5), ...),
        __base__=BaseModel
    )
    # If using Pydantic v2, call model_rebuild to ensure all references are resolved
    if hasattr(DynamicPersonaDatasetQualityReview, "model_rebuild"):
        DynamicPersonaDatasetQualityReview.model_rebuild()

    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": (
                    "You are an expert reviewer evaluating the quality of entries in a dataset of police personas. "
                    "Your task is to assess the quality of the dataset entry itself, not the competency or character of the persona described. "
                    "Rate the following aspects from 0 (worst) to 10 (best):\n"
                    "- clarity: Is the persona description clear and understandable?\n"
                    "- originality: Does the entry avoid clichés and present a unique character?\n"
                    "- coherence: Is the information internally consistent and logically structured?\n"
                    "- diversity: Does the persona contribute to a diverse set of police profiles?\n"
                    "- realism: Does the persona feel plausible and authentic for a police context?\n"
                    "- psychological_depth: Does the entry provide meaningful insight into the persona's inner life, motivations, and psychological complexity? (Focus especially on this metric.)\n"
                    "- consistency: Are details about the persona consistent throughout?\n"
                    "- informativeness: Does the entry provide rich, relevant information about the persona?\n"
                    "- ethical_considerations: Is the entry free from harmful stereotypes or bias?\n"
                    "- demographic_fidelity: Is the persona plausible for the demographic data provided (e.g., a 22 year old should not be described as retiring with decades of experience)?\n"
                    "- overall_score: Your overall assessment of the dataset entry's quality.\n"
                    "Return only the scores in the specified schema, and include a UID string field (UID) for this entry. Remember: you are judging the quality of the dataset entry, not the police persona's job performance."
                ),
            },
            {
                "role": "user",
                "content": row["persona_string"],
            },
        ],
        text_format=DynamicPersonaDatasetQualityReview,
    )

    persona_review = response.output_parsed
    # Ensure the UID is set to the row's UID (in case the model doesn't return it correctly)
    persona_review.UID = uid
    # Upsert into the dictionary using UID as the key
    reviews_dict[uid] = persona_review

# Save the updated dictionary back to disk
import json

In [ ]:
final_reviews_dict = {item: value.model_dump() for item, value in reviews_dict.items()}

In [ ]:
import pickle

with open("openai_list_of_reviews.json", "w", encoding="utf-8") as f:
    json.dump(final_reviews_dict, f, ensure_ascii=False, indent=2)


### Compare models' Cohen-Kappa

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

def compare_raters(openai_path: str, anthropic_path: str):
    # Load JSONs: {persona_id: {feature: score}}
    with open(openai_path) as f:
        openai_data = json.load(f)
    with open(anthropic_path) as f:
        anthropic_data = json.load(f)

    # Build aligned long-form DataFrame
    rows = []
    for pid in set(openai_data) & set(anthropic_data):
        o_feats = openai_data[pid]
        a_feats = anthropic_data[pid]
        for feat in set(o_feats) & set(a_feats):
            if feat != 'UID':
                rows.append({
                    "persona_id": pid,
                    "feature": feat,
                    "openai_score": float(o_feats[feat]),
                    "anthropic_score": float(a_feats[feat]),
                })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No overlapping (persona_id, feature) pairs between raters.")

    # Integer categories for kappa
    df["openai_cat"] = df["openai_score"].round().astype(int)
    df["anthropic_cat"] = df["anthropic_score"].round().astype(int)

    # Per-feature stats
    def feat_stats(g):
        if g["openai_score"].nunique() > 1 and g["anthropic_score"].nunique() > 1:
            r = g["openai_score"].corr(g["anthropic_score"])
        else:
            r = np.nan
        return pd.Series({
            "mean_openai": g["openai_score"].mean(),
            "mean_anthropic": g["anthropic_score"].mean(),
            "mae": (g["openai_score"] - g["anthropic_score"]).abs().mean()
        })

    per_feature = df.groupby("feature").apply(feat_stats).reset_index()

    # Overall stats
    if df["openai_score"].nunique() > 1 and df["anthropic_score"].nunique() > 1:
        overall_r = df["openai_score"].corr(df["anthropic_score"])
    else:
        overall_r = np.nan

    overall = {
        "overall_pearson_r": overall_r,
        "overall_mae": (df["openai_score"] - df["anthropic_score"]).abs().mean(),
    }

    return per_feature, overall

# Example usage:
# per_feature_df, overall_stats = compare_raters("openai_ratings.json", "anthropic_ratings.json")
# print(per_feature_df)
# print(overall_stats)

In [ ]:
per_feature , overall = compare_raters(
    "openai_list_of_reviews.json", "anthropic_persona_reviews.json"
)

In [ ]:
per_feature

In [ ]:
overall